# Lab: Process Multimodal Data with LLMs
**Module 1, Exercise 2 — IBM watsonx Submission**

## Install required libraries

In [ ]:
%%capture
%pip install numpy==2.3.4
%pip install matplotlib==3.10.7
%pip install ibm-watsonx-ai==1.4.7

## Import required libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import json
import os
from PIL import Image

# IBM WatsonX imports
from ibm_watsonx_ai import Credentials
from ibm_watsonx_ai.foundation_models import ModelInference
from ibm_watsonx_ai.metanames import GenTextParamsMetaNames as GenParams
from ibm_watsonx_ai.foundation_models.utils.enums import (
    ModelTypes,
    DecodingMethods,
)

# Suppress warnings
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

## Fetch the data files

In [ ]:
!wget https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/hpTjb6liKBLVHQK0UgMi5A/Recipes.json
!wget https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/fQUs9wQ6aB6ts6fmkD2V2w/Synthetic-User-Reviews.json
!wget https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/5_Rr6ohviItzucyWk6nkrw/synthetic-recipe-images.zip

In [ ]:
import zipfile

with zipfile.ZipFile("synthetic-recipe-images.zip", 'r') as zip_ref:
    zip_ref.extractall()

## Exercise 1: Preprocess the food recipe data
### Step 1: Explore the food recipe JSON file and its images

In [ ]:
### Step 1.1: Load the json file. Define the loaded data as recipe_data.
with open('Recipes.json', 'r') as f:
    recipe_data = json.load(f)

### Step 1.2: Print each key-value pair of the first recipe
first_recipe = recipe_data[0]
for key, value in first_recipe.items():
    print(f"{key} ({type(value).__name__}): {value}")

### Step 1.3: Show the image of the first recipe
img_path = f"synthetic_recipe_images/recipe{first_recipe['id']}.png"
img = Image.open(img_path)
plt.imshow(img)
plt.axis('off')
plt.title(first_recipe['name'])
plt.show()

### Step 2: Define the vision LLM with LLaMA

In [ ]:
import base64

def vision_llm(system_msg, prompt_txt, image_path):
    #system_msg: input system message for the LLM
    #prompt_txt: input user prompt for the LLM
    #image_path: the file path of the input image

    ### Credentials of the model
    model_id = 'meta-llama/llama-4-maverick-17b-128e-instruct-fp8'
    project_id = "skills-network"
    credentials = Credentials(
        url="https://us-south.ml.cloud.ibm.com",
    )
    generate_params = {"max_tokens": 300}

    ### Step 2.1: Define the model by ModelInference
    model = ModelInference(
        model_id=model_id,
        credentials=credentials,
        project_id=project_id,
        params=generate_params
    )

    ### Step 2.2: Encode the input image to a base64 string
    with open(image_path, "rb") as img_file:
        image_b64 = base64.b64encode(img_file.read()).decode("utf-8")

    ### Step 2.3: Define the messages for the model
    messages = [
        {
            "role": "system",
            "content": system_msg
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": prompt_txt
                },
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/jpeg;base64,{image_b64}"
                    }
                }
            ]
        }
    ]

    ### Step 2.4: Get the response for the messages
    response = model.chat(messages=messages)
    return response['choices'][0]['message']['content']

### Step 3: Design and validate prompts for the vision LLM

In [ ]:
### Define the food image caption prompts given a food name.
def image_caption_prompt_template(food_name):
    # food_name: the food name of the recipe

    ### Step 3.1: Design the prompts
    image_caption_system_msg = (
        "You are a professional food photographer and culinary expert. "
        "Your task is to generate concise, accurate, and descriptive captions for food images. "
        "Focus on visible ingredients, cooking style, presentation, colors, and textures."
    )
    image_caption_prompt_txt = (
        f"Please describe this image of {food_name}. "
        "Include details about the visible ingredients, cooking method, presentation style, "
        "colors, textures, and any garnishes. Keep the description concise and informative "
        "in 2-3 sentences."
    )

    return image_caption_system_msg, image_caption_prompt_txt


### Test the prompts on the first recipe
### Step 3.2: Get the prompts with the food name of the first recipe
food_name = recipe_data[0]['name']
system_msg, prompt_txt = image_caption_prompt_template(food_name)

### Step 3.3: Get the test response and print it
img_path = f"synthetic_recipe_images/recipe{recipe_data[0]['id']}.png"
response = vision_llm(system_msg, prompt_txt, img_path)
print(response)

### Step 4: Caption all images and augment the data
**SCREENSHOT: M1L2_caption_all_recipes.jpg**

In [ ]:
### Get captions for each image in the dataset and add to the JSON file
for i in range(len(recipe_data)):
    if (i+1) % 20 == 0:
        print(f'{i+1} out of {len(recipe_data)} is done')

    ### Step 4.1: Get the caption prompts
    food_name = recipe_data[i]['name']
    system_msg, prompt_txt = image_caption_prompt_template(food_name)

    ### Step 4.2: Get the response with the prompts
    img_path = f"synthetic_recipe_images/recipe{recipe_data[i]['id']}.png"
    response = vision_llm(system_msg, prompt_txt, img_path)

    ### Save the response as another item in the recipe data
    recipe_data[i]['image_description'] = response
print('ALL DONE!')

### Step 5: Save the image-caption-augmented recipe data

In [ ]:
filename = 'augmented_food_recipe.json'
with open(filename, 'w', encoding='utf-8') as f:
    json.dump(recipe_data, f, indent=4)
print(f'Saved {len(recipe_data)} recipes to {filename}')

---
## Exercise 2: Preprocess the user visit history
### Step 1: Load the user review data

In [ ]:
### Step 1.1: Load the review dataset with variable name user_review_data
with open('Synthetic-User-Reviews.json', 'r') as f:
    user_review_data = json.load(f)

### Step 1.2: Print the first review by key-value pairs
first_review = user_review_data[0]
for key, value in first_review.items():
    print(f"{key} ({type(value).__name__}): {value}")

### Step 1 (Part 2): Display the first review image

In [ ]:
import ast
import requests

### Step 1.3: Use ast.literal_eval to convert the string list to actual Python list
images_list = ast.literal_eval(user_review_data[0]['images'])

### Step 1.4: Use requests.get() to get the image content
first_image_url = images_list[0]
image_response = requests.get(first_image_url)

### Step 1.5: Write the image content to a temporary file
with open('review_image_placeholder.jpg', 'wb') as img_file:
    img_file.write(image_response.content)

### Step 1.6: Open and show the image
img = Image.open('review_image_placeholder.jpg')
plt.imshow(img)
plt.axis('off')
plt.title('First Review Image')
plt.show()

### Step 2: Define the prompt and validation

In [ ]:
### Prompt template: caption the images with the context of the reviews
def review_context_image_caption_prompt_template(reviews):
    # reviews: the written review content

    ### Step 2.1: Design your prompts
    review_context_image_caption_system_msg = (
        "You are a culinary expert and food critic. "
        "Your task is to generate concise image descriptions for food images "
        "that align with the context and sentiment expressed in customer reviews. "
        "Focus on visual details such as food presentation, ingredients, colors, and textures "
        "that match the experience described in the reviews."
    )
    review_context_image_caption_prompt_txt = (
        f'Based on the following customer review(s): "{reviews}", '
        "please describe this food image in 2-3 sentences. "
        "Highlight visual details that are consistent with the reviewer's experience and sentiment."
    )

    return review_context_image_caption_system_msg, review_context_image_caption_prompt_txt


### Step 2.2: Get the prompts
first_review_text = user_review_data[0]['text']
system_msg, prompt_txt = review_context_image_caption_prompt_template(first_review_text)

### Step 2.3: Get the response by the vision_llm
response = vision_llm(system_msg, prompt_txt, 'review_image_placeholder.jpg')
print(response)

### Step 3: Caption all review images and augment the data

In [ ]:
from tenacity import retry, stop_after_attempt, wait_exponential

### URL Request function with Retry
@retry(stop=stop_after_attempt(10), wait=wait_exponential(multiplier=1, min=1, max=10))
def get_data_with_retry(url):
    response = requests.get(url, timeout=5)
    response.raise_for_status()
    return response

### Start the for loop
for i in range(len(user_review_data)):
    ### Step 3.1: Convert the string to the Python list of image urls
    review_images = ast.literal_eval(user_review_data[i]['images'])

    review_image_captions = []
    if len(review_images) > 0:
        for img_url in review_images:
            try:
                ### Step 3.2: Use get_data_with_retry to get the image_data
                image_data = get_data_with_retry(img_url)
                print("Success!")
            except Exception as e:
                print(f"All retries failed at url {img_url}:", e)
                continue
            image = image_data.content
            with open('review_image_placeholder.jpg', 'wb') as img_file:
                img_file.write(image)

            ### Step 3.3: Get the prompts, get the response, and append
            review_text = user_review_data[i]['text']
            sys_msg, prompt_txt = review_context_image_caption_prompt_template(review_text)
            caption = vision_llm(sys_msg, prompt_txt, 'review_image_placeholder.jpg')
            review_image_captions.append(caption)

    ### Append the review_image_captions to the review data
    user_review_data[i]['image_captions'] = review_image_captions
print('ALL DONE!')

### Step 4: Save the image-caption-augmented user review data

In [ ]:
filename = 'augmented_user_review.json'
with open(filename, 'w', encoding='utf-8') as f:
    json.dump(user_review_data, f, indent=4)
print(f'Saved {len(user_review_data)} reviews to {filename}')